# Triedros e ângulos poliédricos

**Módulo 06 — Geometria Espacial** · Notebook 06 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0606a_triedros_e_angulos_poliedricos.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0603a_perpendicularidade.ipynb`](0603a_perpendicularidade.ipynb) (retas perpendiculares a planos, planos perpendiculares) e [`0605a_diedros.ipynb`](0605a_diedros.ipynb) (medida de diedros, seção reta, seções igualmente inclinadas)
- Módulo 04 — Geometria Plana: [`0404a_triangulos_conceito_elementos_e_classificacao.ipynb`](../04_geometria_plana/0404a_triangulos_conceito_elementos_e_classificacao.ipynb) (desigualdade triangular), [`0405a_congruencia_de_triangulos.ipynb`](../04_geometria_plana/0405a_congruencia_de_triangulos.ipynb) (casos de congruência de triângulos) e [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (polígonos regulares e seus ângulos internos)
- Módulo 05 — Trigonometria: [`0510a_triangulos_quaisquer.ipynb`](../05_trigonometria/0510a_triangulos_quaisquer.ipynb) (lei dos cossenos)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- identificar os seis elementos de um triedro (três faces e três diedros);
- decidir se três ângulos podem ser as faces de um triedro;
- aplicar os casos de congruência de triedros e as propriedades do triedro polar;
- reconhecer ângulos poliédricos convexos e usar a condição "soma das faces menor que 360°".

**Tempo estimado:** 70 a 90 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets
import math

## Dá para fechar esse canto?

Pegue uma folha de papel, recorte três ângulos (três "fatias de pizza" com a ponta em comum) e tente colá-los lado a lado para formar uma **ponta**, como o canto de uma caixa.

- Com três ângulos de 90°, funciona: é exatamente o canto de uma caixa de sapatos.
- Com três ângulos de 60°, também: vira a ponta de uma pirâmide.
- Agora tente com **100°, 50° e 40°**. Não fecha! O ângulo de 100° é "largo demais" para os outros dois.
- E com **130°, 120° e 110°**? Os três juntos ocupam uma volta inteira (360°). Também não fecha.

🕹️ No widget abaixo, escolha os três ângulos. Em cima está o papel aberto (a **planificação**); embaixo, a ponta montada, quando ela existe.

In [2]:
# As coordenadas (x, y, z) abaixo servem só para posicionar os desenhos.
CORES_DAS_FACES = [
    u3d.CORES["plano"], u3d.CORES["plano_secundario"], u3d.CORES["plano_terciario"]
]


def desenhar_planificacao(
    face_a: float, face_b: float, face_c: float, titulo: str
) -> None:
    """Desenha, vista de cima, as três faces abertas no papel em volta de V."""
    p = u3d.planificacao_do_triedro(face_a, face_b, face_c)
    V = p["V"]
    fig = u3d.criar_figura(titulo=titulo)
    # As faces na ordem em que o papel foi aberto: AVB (f_c), BVC (f_a), CVA' (f_b).
    faces_abertas = [("A", "B"), ("B", "C"), ("C", "A_linha")]
    for (inicio, fim), cor in zip(faces_abertas, CORES_DAS_FACES, strict=True):
        fig = u3d.desenhar_poligono(
            fig, [V, p[inicio], p[fim]], cor=cor, opacidade=0.35
        )
    for nome, rotulo in [("A", "a"), ("B", "b"), ("C", "c"), ("A_linha", "a'")]:
        fig = u3d.desenhar_segmento(fig, V, p[nome])
        # O nome fica um pouco além da ponta da aresta (V está na origem).
        fig = u3d.adicionar_rotulo(fig, tuple(1.15 * c for c in p[nome]), rotulo)

    # A folga: o quanto falta para as três faces completarem uma volta inteira.
    soma = face_a + face_b + face_c
    folga = 360 - soma
    if folga > 0:
        # A folga pode passar de 180°, e um arco só vai até 180°: desenhamos o arco
        # em duas metades, passando por um ponto M no meio do caminho de A' até A.
        # Seno e cosseno (módulo 05) servem só para posicionar o ponto M.
        meio = math.radians(360 - folga / 2)
        M = (math.cos(meio), math.sin(meio), 0)
        fig = u3d.desenhar_angulo(
            fig, V, p["A_linha"], M, cor=u3d.CORES["apoio"], raio=1
        )
        fig = u3d.desenhar_angulo(
            fig, V, M, p["A"], rotulo=f"folga: {folga}°", cor=u3d.CORES["apoio"],
            raio=1,
        )
    fig = u3d.desenhar_ponto(fig, V, rotulo="V")
    fig = u3d.definir_camera(fig, olho=(0, 0, 2.5))
    u3d.mostrar(fig)

    if folga > 0:
        print(f"Folga: 360° − {soma}° = {folga}°")
    elif folga == 0:
        print("Soma de 360°: as faces ocupam uma volta inteira e o papel fica plano.")
    else:
        print(
            f"Os ângulos se sobrepõem: juntos, passam {-folga}° de uma volta inteira."
        )


def montar_canto(angulo_1: int, angulo_2: int, angulo_3: int) -> None:
    """Mostra o papel aberto e, quando ela existe, a ponta montada."""
    desenhar_planificacao(
        angulo_1, angulo_2, angulo_3, titulo="O papel aberto (vista de cima)"
    )
    try:
        t = u3d.vertices_do_triedro(angulo_1, angulo_2, angulo_3)
    except ValueError as erro:
        print(f"Não fecha! {erro}")
        return
    fig = u3d.criar_figura(titulo="A ponta montada")
    fig = u3d.desenhar_triedro(fig, t["V"], t["A"], t["B"], t["C"])
    u3d.mostrar(fig)
    print("Fecha! As três faces formam uma ponta.")


interact(
    montar_canto,
    angulo_1=widgets.IntSlider(
        min=10, max=170, step=10, value=90, description="ângulo 1 (°)",
        continuous_update=False,
    ),
    angulo_2=widgets.IntSlider(
        min=10, max=170, step=10, value=90, description="ângulo 2 (°)",
        continuous_update=False,
    ),
    angulo_3=widgets.IntSlider(
        min=10, max=170, step=10, value=90, description="ângulo 3 (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=90, continuous_update=False, description='ângulo 1 (°)', max=170, min=10…

A ponta que você montou se chama **triedro**. Repare na folga: quando ela existe, as faces "dobram" para cima e se fecham; quando ela some (a soma chega a 360°), o papel fica plano e não há ponta. Neste notebook você vai descobrir exatamente quais ângulos fecham uma ponta, e no próximo notebook vai usar isso para descobrir por que só existem **cinco** poliedros regulares.

## 1. Triedro: conceitos e elementos

> 📌 **Definição.** Dadas três semirretas $a$, $b$ e $c$, de mesma origem $V$ e **não coplanares**, o **triedro** $V(a, b, c)$ é a figura formada pelos três ângulos $a\hat{V}b$, $b\hat{V}c$ e $a\hat{V}c$.

**Elementos**
- **Vértice:** o ponto $V$.
- **Arestas:** as semirretas $a$, $b$, $c$.
- **Faces:** os três ângulos. Cada face recebe o nome da aresta oposta a ela:
  $f_a = b\hat{V}c$, $f_b = a\hat{V}c$, $f_c = a\hat{V}b$.
- **Diedros:** os três diedros formados pelas faces, um em cada aresta: $d_a$, $d_b$, $d_c$.

Ao todo, o triedro tem **seis elementos**: três faces e três diedros. É como um triângulo, que tem três lados e três ângulos.

| Triângulo | Triedro |
|---|---|
| 3 vértices | 1 vértice e 3 arestas |
| 3 lados | 3 faces |
| 3 ângulos | 3 diedros |

Um triedro com as três faces retas é chamado **trirretângulo**. Ele está em todo lugar: no canto de uma sala, no vértice de um cubo, na quina de uma caixa.

> 💡 **Dica.** Não confunda **face** e **diedro**. A face é um ângulo **plano**, medido dentro de uma das faces (por exemplo, o ângulo do canto de uma parede). O diedro é a abertura **entre duas faces**, medida pela seção reta (notebook 05).

🕹️ Escolha um dos triedros abaixo, que aparecem em objetos do dia a dia, e compare os seis elementos.

In [ ]:
# Faces (f_a, f_b, f_c) de cada triedro, em graus.
TRIEDROS_FAMOSOS = {
    "Canto de uma caixa (vértice do cubo)": (90, 90, 90),
    "Ponta de uma pirâmide de faces equiláteras (tetraedro regular)": (60, 60, 60),
    "Canto de uma barra de chocolate triangular (prisma)": (90, 90, 60),
    "Vértice de uma bola de 12 faces pentagonais (dodecaedro regular)": (
        108, 108, 108,
    ),
}


def explorar_triedro_famoso(triedro: str) -> None:
    """Desenha o triedro escolhido e mostra os seus seis elementos."""
    t = u3d.vertices_do_triedro(*TRIEDROS_FAMOSOS[triedro])
    V, A, B, C = t["V"], t["A"], t["B"], t["C"]
    elementos = u3d.medir_elementos_do_triedro(V, A, B, C)
    faces, diedros = elementos["faces"], elementos["diedros"]

    fig = u3d.criar_figura(titulo=triedro)
    fig = u3d.desenhar_triedro(fig, V, A, B, C)
    # Cada arco marca uma face, com o nome da aresta oposta a ela.
    for nome, (P, Q) in {"a": (B, C), "b": (A, C), "c": (A, B)}.items():
        fig = u3d.desenhar_angulo(fig, V, P, Q, rotulo=f"f_{nome}", raio=1)
    u3d.mostrar(fig)

    print(f"{'':10}{'aresta a':>11}{'aresta b':>11}{'aresta c':>11}")
    print(f"{'face':10}" + "".join(f"{faces[x]:>10.1f}°" for x in "abc"))
    print(f"{'diedro':10}" + "".join(f"{diedros[x]:>10.1f}°" for x in "abc"))
    print()
    soma_das_faces = faces["a"] + faces["b"] + faces["c"]
    soma_dos_diedros = diedros["a"] + diedros["b"] + diedros["c"]
    print(f"Soma das faces: {soma_das_faces:.1f}°")
    print(f"Soma dos diedros: {soma_dos_diedros:.1f}°")


interact(
    explorar_triedro_famoso,
    triedro=widgets.Dropdown(
        options=list(TRIEDROS_FAMOSOS), description="triedro",
        layout=widgets.Layout(width="600px"),
    ),
);

interactive(children=(Dropdown(description='triedro', layout=Layout(width='600px'), options=('Canto de uma cai…

**Pare e pense**

1. No vértice $A$ do cubo, com arestas $AB$, $AD$ e $AE$, qual é a face oposta à aresta $AE$? Quanto ela mede?
2. Na barra de chocolate triangular, por que o diedro da aresta vertical mede 60°, igual à face da base?

<details>
<summary>Ver respostas</summary>

1. É o ângulo $B\hat{A}D$, na base. Mede 90°.
2. A aresta vertical é perpendicular à base. Então a base é uma seção reta do diedro dessa aresta, e a seção reta é exatamente a face de 60°.
</details>

## 2. Relações entre as faces

O gancho mostrou que nem todo trio de ângulos forma um triedro. As duas condições são:

**Teorema 1 (desigualdade das faces)**
> Em todo triedro, cada face é menor que a soma das outras duas:
> $$f_a < f_b + f_c, \qquad f_b < f_a + f_c, \qquad f_c < f_a + f_b$$

É a "desigualdade triangular" dos triedros.

**Teorema 2 (soma das faces)**
> Em todo triedro, a soma das faces é menor que 360°:
> $$f_a + f_b + f_c < 360°$$

**E vale a volta:** se três ângulos entre 0° e 180° satisfazem essas duas condições, então existe um triedro com essas faces.

<details>
<summary>Ver a demonstração do Teorema 2 (usando o Teorema 1)</summary>

Corte o triedro por um plano que encontra as arestas em $A$, $B$ e $C$. Aparecem três triângulos laterais, $VAB$, $VBC$ e $VCA$, e o triângulo $ABC$ na base.

- A soma dos ângulos dos três triângulos laterais é $3 \cdot 180° = 540°$. Essa soma é igual a (soma das faces em $V$) + (soma dos seis ângulos da "base", dois em cada um dos pontos $A$, $B$, $C$).
- Em $A$ existe um triedro com faces $V\hat{A}B$, $V\hat{A}C$ e $B\hat{A}C$. Pelo Teorema 1: $B\hat{A}C < V\hat{A}B + V\hat{A}C$. O mesmo vale em $B$ e em $C$.
- Somando: $B\hat{A}C + A\hat{B}C + A\hat{C}B < $ (soma dos seis ângulos da base). Mas o lado esquerdo é a soma dos ângulos do triângulo $ABC$, que é 180°.
- Então a soma dos seis ângulos da base é maior que 180°, e a soma das faces em $V$ é menor que $540° - 180° = 360°$. ∎
</details>

<details>
<summary>Ver a ideia da demonstração do Teorema 1</summary>

Seja $f_a = b\hat{V}c$ a maior face (para as outras, a desigualdade é imediata). Dentro de $f_a$, trace a semirreta $Vd$ com $b\hat{V}d = f_c$. Marque $A$ em $a$ e $D$ em $Vd$ com $VA = VD$, e trace uma reta que corta $b$ em $B$, $Vd$ em $D$ e $c$ em $C$.

Os triângulos $AVB$ e $DVB$ são congruentes (LAL), então $AB = DB$. No triângulo $ABC$: $AC > BC - AB = BC - DB = DC$. Os triângulos $AVC$ e $DVC$ têm $VA = VD$ e $VC$ em comum, mas $AC > DC$. Quando dois triângulos têm dois lados respectivamente iguais, o maior terceiro lado fica oposto ao maior ângulo; logo $f_b = A\hat{V}C > D\hat{V}C = f_a - f_c$. Isto é, $f_a < f_b + f_c$. ∎
</details>

🕹️ Use o widget abaixo como um "verificador de triedros": ele confere as condições e, quando o triedro existe, mostra também os seus diedros.

In [ ]:
def verificar_triedro(face_a: int, face_b: int, face_c: int) -> None:
    """Confere as condições de existência e desenha o triedro (ou a planificação)."""
    soma = face_a + face_b + face_c
    condicoes = {
        f"f_a < f_b + f_c  ({face_a} < {face_b + face_c})": face_a < face_b + face_c,
        f"f_b < f_a + f_c  ({face_b} < {face_a + face_c})": face_b < face_a + face_c,
        f"f_c < f_a + f_b  ({face_c} < {face_a + face_b})": face_c < face_a + face_b,
        f"soma < 360°      ({soma} < 360)": soma < 360,
    }
    for descricao, vale in condicoes.items():
        print(f"{'✔' if vale else '✘'} {descricao}")
    print()

    if all(condicoes.values()):
        t = u3d.vertices_do_triedro(face_a, face_b, face_c)
        fig = u3d.criar_figura(titulo="O triedro existe")
        fig = u3d.desenhar_triedro(fig, t["V"], t["A"], t["B"], t["C"])
        u3d.mostrar(fig)
        diedros = u3d.medir_elementos_do_triedro(t["V"], t["A"], t["B"], t["C"])[
            "diedros"
        ]
        print("O triedro existe. Os seus diedros (conferidos com o utils_3d):")
        for x in "abc":
            print(f"  d_{x} = {diedros[x]:.1f}°")
    else:
        print("Essas faces não formam um triedro.")
        desenhar_planificacao(
            face_a, face_b, face_c, titulo="A planificação não fecha"
        )


interact(
    verificar_triedro,
    face_a=widgets.IntSlider(
        min=10, max=170, step=5, value=80, description="f_a (°)",
        continuous_update=False,
    ),
    face_b=widgets.IntSlider(
        min=10, max=170, step=5, value=70, description="f_b (°)",
        continuous_update=False,
    ),
    face_c=widgets.IntSlider(
        min=10, max=170, step=5, value=60, description="f_c (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=80, continuous_update=False, description='f_a (°)', max=170, min=10, ste…

**Pare e pense**

1. Existe triedro com faces 100°, 120° e 150°? E com 30°, 40° e 80°?
2. Num triedro, duas faces medem 70° e 50°. Entre que valores pode estar a terceira face?

<details>
<summary>Ver respostas</summary>

1. Não nos dois casos: no primeiro, a soma é 370° > 360°; no segundo, 80° > 30° + 40° = 70°.
2. A terceira face $x$ precisa satisfazer $x < 70 + 50 = 120$ e $70 < 50 + x$, ou seja, $x > 20$. A soma $120 + x < 360$ já fica garantida. Então $20° < x < 120°$.
</details>

## 3. Congruência de triedros

> 📌 **Definição.** Dois triedros são **congruentes** quando é possível fazer uma correspondência entre suas arestas de modo que as faces correspondentes e os diedros correspondentes sejam congruentes.

Assim como nos triângulos, não é preciso conferir os seis elementos. Bastam três, em certas posições:

| Caso | O que é preciso conhecer | Análogo nos triângulos |
|---|---|---|
| **FDF** | duas faces e o diedro entre elas | LAL |
| **DFD** | dois diedros e a face entre eles | ALA |
| **FFF** | as três faces | LLL |
| **DDD** | os três diedros | **não tem análogo!** |

> ⚠️ **Erro comum.** Achar que o caso DDD não vale, por analogia com os triângulos. Dois triângulos com os três ângulos iguais podem ter tamanhos diferentes (são apenas semelhantes). Mas um triedro **não tem tamanho**: suas arestas são semirretas infinitas. Por isso, três diedros iguais garantem triedros congruentes. A demonstração usa o triedro polar (seção 4).

> 💡 **Dica.** Triedros congruentes podem ser "espelhados", como a mão direita e a mão esquerda: têm os mesmos seis elementos, mas não dá para encaixar um no outro só girando. A definição de congruência inclui esse caso.

<details>
<summary>Ver a demonstração do caso FFF</summary>

Sejam $V(a, b, c)$ e $V'(a', b', c')$ com as faces correspondentes congruentes. Vamos mostrar que o diedro $d_a$ é congruente a $d_{a'}$ (os outros são análogos). Para simplificar, suponha as faces $f_b$ e $f_c$ agudas; o caso geral segue a mesma ideia, com alguns cuidados extras.

1. Marque $P$ em $a$ e $P'$ em $a'$ com $VP = V'P'$.
2. Nas faces que contêm $a$, trace por $P$ as perpendiculares a $a$, cortando $b$ em $M$ e $c$ em $N$. Faça o mesmo em $V'$, obtendo $M'$ e $N'$. O ângulo $M\hat{P}N$ é uma **seção reta** de $d_a$ (notebook 05).
3. Os triângulos $VPM$ e $V'P'M'$ são congruentes (ALA: ângulo reto em $P$, lado $VP$, ângulo $f_c$ em $V$). Então $PM = P'M'$ e $VM = V'M'$. Do mesmo modo, $PN = P'N'$ e $VN = V'N'$.
4. Os triângulos $VMN$ e $V'M'N'$ são congruentes (LAL, com a face $f_a$ entre $VM$ e $VN$). Então $MN = M'N'$.
5. Os triângulos $PMN$ e $P'M'N'$ têm os três lados iguais (LLL), então $M\hat{P}N = M'\hat{P'}N'$. As seções retas são iguais, logo $d_a = d_{a'}$. ∎

A seção reta é uma seção igualmente inclinada (com inclinação de 90°). É por isso que o notebook 05 dizia que essas seções seriam a ferramenta para comparar triedros.
</details>

🕹️ **Veja o caso FFF em ação.** O widget monta dois triedros com as mesmas faces, em posições diferentes (e, se quiser, espelhado). Compare os diedros.

In [ ]:
DESLOCAMENTO = 7  # distância entre os dois desenhos, só para não se sobreporem


def comparar_pelo_caso_fff(
    face_a: int, face_b: int, face_c: int, espelhar: bool
) -> None:
    """Desenha dois triedros com as mesmas faces e compara os seis elementos."""
    try:
        t1 = u3d.vertices_do_triedro(face_a, face_b, face_c)
    except ValueError as erro:
        print(erro)
        return

    # Triedro 2: a mesma construção, mas começando por outra aresta. Nela, a face
    # oposta ao ponto "A" é face_b, a oposta ao "B" é face_c e a oposta ao "C" é
    # face_a; renomeamos os pontos para que cada aresta a', b', c' fique de frente
    # para a face certa. O triedro fica girado em relação ao primeiro.
    t = u3d.vertices_do_triedro(face_b, face_c, face_a)
    t2 = {"V": t["V"], "A": t["C"], "B": t["A"], "C": t["B"]}
    # Inverter o z só para desenhar o espelho, como a mão esquerda e a direita; e
    # somar DESLOCAMENTO ao x só para o segundo desenho ficar ao lado do primeiro.
    sinal = -1 if espelhar else 1
    t2 = {nome: (x + DESLOCAMENTO, y, sinal * z) for nome, (x, y, z) in t2.items()}

    fig = u3d.criar_figura(titulo="Mesmas faces, posições diferentes")
    fig = u3d.desenhar_triedro(fig, t1["V"], t1["A"], t1["B"], t1["C"])
    fig = u3d.desenhar_triedro(
        fig, t2["V"], t2["A"], t2["B"], t2["C"], rotulos_arestas=("a'", "b'", "c'"),
        rotulo_vertice="V'",
    )
    fig = u3d.definir_camera(fig, olho=(0.3, -2.2, 1.0))
    u3d.mostrar(fig)

    e1 = u3d.medir_elementos_do_triedro(t1["V"], t1["A"], t1["B"], t1["C"])
    e2 = u3d.medir_elementos_do_triedro(t2["V"], t2["A"], t2["B"], t2["C"])
    print(f"{'':16}{'triedro 1':>12}{'triedro 2':>12}")
    for tipo, nome_do_tipo in (("faces", "face"), ("diedros", "diedro")):
        for x in "abc":
            rotulo = f"{nome_do_tipo} {x} / {x}'"
            print(f"{rotulo:16}{e1[tipo][x]:>11.1f}°{e2[tipo][x]:>11.1f}°")
    print()
    print("Faces iguais ⟹ diedros iguais: os triedros são congruentes (caso FFF).")


interact(
    comparar_pelo_caso_fff,
    face_a=widgets.IntSlider(
        min=20, max=160, step=10, value=70, description="f_a (°)",
        continuous_update=False,
    ),
    face_b=widgets.IntSlider(
        min=20, max=160, step=10, value=80, description="f_b (°)",
        continuous_update=False,
    ),
    face_c=widgets.IntSlider(
        min=20, max=160, step=10, value=100, description="f_c (°)",
        continuous_update=False,
    ),
    espelhar=widgets.Checkbox(value=False, description="Espelhar o segundo triedro"),
);

interactive(children=(IntSlider(value=70, continuous_update=False, description='f_a (°)', max=160, min=20, ste…

**Pare e pense**

1. Dois triedros têm faces 60°, 60° e 60°. O que se pode dizer dos seus diedros?
2. Por que o caso "AAA" não serve para triângulos, mas o caso DDD serve para triedros?

<details>
<summary>Ver respostas</summary>

1. São iguais (caso FFF): todos medem cerca de 70,5°.
2. Triângulos têm tamanho, e ampliar um triângulo mantém os ângulos. Triedros não têm tamanho (as arestas são infinitas), então não existe "triedro ampliado": diedros iguais já obrigam os triedros a serem congruentes.
</details>

## 4. Triedros polares (ou suplementares)

> 📌 **Definição.** Dado um triedro $V(a, b, c)$, o seu **triedro polar** $V(a', b', c')$ tem o mesmo vértice e arestas:
> - $a'$ perpendicular ao plano da face $f_a$ ($b\hat{V}c$), do mesmo lado que $a$;
> - $b'$ perpendicular ao plano da face $f_b$, do mesmo lado que $b$;
> - $c'$ perpendicular ao plano da face $f_c$, do mesmo lado que $c$.

**Teorema**
> As faces do triedro polar são os **suplementos** dos diedros do triedro original, e os diedros do polar são os suplementos das faces do original:
> $$f'_a = 180° - d_a, \qquad d'_a = 180° - f_a \qquad \text{(e o mesmo para } b \text{ e } c\text{)}$$
> Além disso, o polar do polar é o triedro original.

**Consequências para os diedros.** Aplicando as relações das faces (seção 2) ao triedro polar e trocando cada face por $180°$ menos um diedro:
- $d_a + d_b + d_c > 180°$ (a soma dos diedros é **maior** que 180°);
- $d_a + d_b + d_c < 540°$;
- $d_b + d_c < 180° + d_a$ (e as outras duas parecidas).

Também sai daqui o caso **DDD**: se dois triedros têm os mesmos diedros, seus polares têm as mesmas faces; pelo caso FFF, os polares são congruentes; então os triedros originais (os polares dos polares) também são.

🎯 **Aplicação: triângulos na superfície da Terra.** Coloque o vértice de um triedro no **centro** da Terra. As três faces cortam a superfície num **triângulo esférico**: os lados do triângulo correspondem às faces, e os ângulos do triângulo correspondem aos diedros. Por isso, a soma dos ângulos de um triângulo desenhado sobre a Terra é sempre **maior que 180°**! Pense no triângulo com vértices no Polo Norte e em dois pontos do Equador separados por 90° de longitude: os três ângulos são retos, e a soma dá 270°.

> 🕵️ **Curiosidade histórica.** A trigonometria dos triângulos esféricos foi desenvolvida por astrônomos e matemáticos gregos, indianos e do mundo islâmico medieval, que precisavam calcular posições de estrelas e direções na superfície da Terra. Um dos usos mais conhecidos era determinar a direção de Meca a partir de qualquer cidade. Mais tarde, essa trigonometria se tornou fundamental para a navegação marítima.

🕹️ Veja um triedro e o seu polar. Mexa nas faces do original e compare as duas tabelas.

In [6]:
def explorar_triedro_polar(
    face_a: int, face_b: int, face_c: int, mostrar_angulos_retos: bool
) -> None:
    """Desenha um triedro e o seu polar e confere o teorema dos suplementos."""
    try:
        t = u3d.vertices_do_triedro(face_a, face_b, face_c)
    except ValueError as erro:
        print(erro)
        return
    V = t["V"]
    polar = u3d.triedro_polar(V, t["A"], t["B"], t["C"])

    fig = u3d.criar_figura(titulo="Um triedro e o seu polar")
    fig = u3d.desenhar_triedro(fig, V, t["A"], t["B"], t["C"])
    fig = u3d.desenhar_triedro(
        fig, V, polar["A"], polar["B"], polar["C"], mostrar_faces=False,
        rotulos_arestas=("a'", "b'", "c'"), rotulo_vertice=None,
        cor_arestas=u3d.CORES["destaque"],
    )
    if mostrar_angulos_retos:
        # a' forma 90° com b e com c (as arestas da face f_a); o mesmo para b' e c'.
        for linha, outras in (("A", "BC"), ("B", "AC"), ("C", "AB")):
            for outra in outras:
                fig = u3d.desenhar_angulo_reto(fig, V, polar[linha], t[outra])
    u3d.mostrar(fig)

    e = u3d.medir_elementos_do_triedro(V, t["A"], t["B"], t["C"])
    ep = u3d.medir_elementos_do_triedro(V, polar["A"], polar["B"], polar["C"])
    f, d = e["faces"], e["diedros"]
    f_polar, d_polar = ep["faces"], ep["diedros"]
    # Colunas: os elementos do original, os do polar e as contas do teorema.
    print(f"{'':10}{'original':^18}{'polar':^18}{'contas':^18}")
    titulos = ("f", "d", "f'", "d'", "180 − d", "180 − f")
    print(f"{'':10}" + "".join(f"{titulo:>9}" for titulo in titulos))
    for x in "abc":
        valores = (f[x], d[x], f_polar[x], d_polar[x], 180 - d[x], 180 - f[x])
        print(f"{f'aresta {x}':10}" + "".join(f"{v:>8.1f}°" for v in valores))
    print("Repare: f' = 180 − d e d' = 180 − f, em todas as arestas.")

    print()
    soma_dos_diedros = d["a"] + d["b"] + d["c"]
    print(
        f"Soma dos diedros: {soma_dos_diedros:.1f}° "
        "(maior que 180° e menor que 540°)"
    )
    for x, y, z in ("abc", "bca", "cab"):
        print(f"d_{y} + d_{z} = {d[y] + d[z]:.1f}° < 180° + d_{x} = {180 + d[x]:.1f}°")


interact(
    explorar_triedro_polar,
    face_a=widgets.IntSlider(
        min=30, max=150, step=10, value=70, description="f_a (°)",
        continuous_update=False,
    ),
    face_b=widgets.IntSlider(
        min=30, max=150, step=10, value=80, description="f_b (°)",
        continuous_update=False,
    ),
    face_c=widgets.IntSlider(
        min=30, max=150, step=10, value=100, description="f_c (°)",
        continuous_update=False,
    ),
    mostrar_angulos_retos=widgets.Checkbox(
        value=False, description="Mostrar ângulos retos"
    ),
);

interactive(children=(IntSlider(value=70, continuous_update=False, description='f_a (°)', max=150, min=30, ste…

**Pare e pense**

1. Um triedro tem diedros de 70°, 80° e 100°. Quais são as faces do seu polar?
2. O triedro trirretângulo é polar de si mesmo. Por quê?

<details>
<summary>Ver respostas</summary>

1. $180 - 70 = 110°$, $180 - 80 = 100°$ e $180 - 100 = 80°$.
2. No triedro trirretângulo, cada aresta já é perpendicular ao plano das outras duas. Então $a' = a$, $b' = b$ e $c' = c$.
</details>

## 5. Ângulos poliédricos convexos

Um triedro tem três arestas. A ponta de uma pirâmide de base quadrada tem **quatro**; o vértice de um icosaedro (o "dado de 20 faces") tem **cinco**. A figura geral é o **ângulo poliédrico**.

> 📌 **Definição.** Dadas $n \ge 3$ semirretas $Va_1, Va_2, \ldots, Va_n$ de mesma origem $V$, tais que três consecutivas quaisquer não são coplanares, o **ângulo poliédrico** $V(a_1, a_2, \ldots, a_n)$ é a figura formada pelos ângulos $a_1\hat{V}a_2$, $a_2\hat{V}a_3$, ..., $a_n\hat{V}a_1$ (as **faces**). Ele é **convexo** quando o plano de cada face deixa todas as outras arestas do mesmo lado.

Os elementos são os mesmos do triedro: vértice, $n$ arestas, $n$ faces e $n$ diedros. O triedro é o ângulo poliédrico com $n = 3$.

**Propriedades (ângulo poliédrico convexo)**
1. Cada face é menor que a soma das outras.
2. **A soma das faces é menor que 360°.**
3. A soma dos diedros está entre $(n-2) \cdot 180°$ e $n \cdot 180°$.

A demonstração da propriedade 2 é igual à do triedro: corte o ângulo poliédrico por um plano, obtendo um polígono de $n$ lados, e use a relação das faces nos triedros que aparecem em cada vértice do polígono.

> ⚠️ **Erro comum.** Esquecer que a propriedade 2 vale para ângulos poliédricos **convexos**. Num ângulo poliédrico "amassado para dentro" (não convexo), a soma das faces pode passar de 360°.

**O "defeito" e o papel.** A diferença $360° - (\text{soma das faces})$ é o quanto "falta" para o papel ficar plano. Para fazer um chapéu de festa ou o canto de uma caixa, você **tira** um pedaço do papel: é esse defeito que faz a superfície se curvar e formar uma ponta.

> 🕵️ **Curiosidade histórica.** O filósofo e matemático francês **René Descartes** (século XVII) descobriu que, em qualquer poliedro convexo, a soma dos defeitos de todos os vértices é sempre a mesma: **720°**. Um cubo tem 8 vértices com defeito de 90° cada (8 × 90° = 720°); um tetraedro tem 4 vértices com defeito de 180° cada (4 × 180° = 720°). Esse resultado está ligado à famosa Relação de Euler, que você verá no próximo notebook.

**Polígonos regulares em volta de um vértice.** Quantas faces iguais, de um mesmo polígono regular, podem se encontrar num vértice de um ângulo poliédrico convexo? 🕹️ Mexa no widget e depois veja a tabela.

In [7]:
def explorar_angulo_poliedrico(numero_de_arestas: int, face: int) -> None:
    """Desenha o ângulo poliédrico regular ou, se ele não existe, as faces abertas."""
    n = numero_de_arestas
    soma = n * face
    defeito = 360 - soma
    print(f"Soma das faces: {n} × {face}° = {soma}°")
    print(f"Defeito: 360° − {soma}° = {defeito}°")

    try:
        vertice, pontos = u3d.vertices_do_angulo_poliedrico_regular(n, face)
    except ValueError as erro:
        print(f"Não existe! {erro}")
        # As n faces abertas no papel, uma ao lado da outra, em volta de V.
        # Seno e cosseno (módulo 05) servem só para girar os pontos em volta de V.
        fig = u3d.criar_figura(titulo="As faces abertas no papel (vista de cima)")
        for k in range(n):
            t1, t2 = math.radians(k * face), math.radians((k + 1) * face)
            P1 = (3 * math.cos(t1), 3 * math.sin(t1), 0)
            P2 = (3 * math.cos(t2), 3 * math.sin(t2), 0)
            fig = u3d.desenhar_poligono(
                fig, [(0, 0, 0), P1, P2], cor=CORES_DAS_FACES[k % 3], opacidade=0.3
            )
        fig = u3d.desenhar_ponto(fig, (0, 0, 0), rotulo="V")
        fig = u3d.definir_camera(fig, olho=(0, 0, 2.5))
        u3d.mostrar(fig)
        return

    fig = u3d.criar_figura(titulo=f"Ângulo poliédrico regular com {n} arestas")
    fig = u3d.desenhar_angulo_poliedrico(fig, vertice, pontos)
    u3d.mostrar(fig)

    # Cada diedro tem como aresta uma aresta do ângulo poliédrico, e as faces
    # passam pelas arestas vizinhas (a anterior e a seguinte).
    diedros = [
        u3d.medir_angulo_do_diedro(
            vertice, pontos[i], pontos[i - 1], pontos[(i + 1) % n]
        )
        for i in range(n)
    ]
    soma_dos_diedros = sum(diedros)
    print(f"Cada diedro (conferido com o utils_3d): {diedros[0]:.1f}°")
    print(
        f"Soma dos diedros: {soma_dos_diedros:.1f}° "
        f"(entre {(n - 2) * 180}° e {n * 180}°)"
    )


interact(
    explorar_angulo_poliedrico,
    numero_de_arestas=widgets.IntSlider(
        min=3, max=8, step=1, value=4, description="arestas",
        continuous_update=False,
    ),
    face=widgets.IntSlider(
        min=10, max=170, step=5, value=60, description="face (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=4, continuous_update=False, description='arestas', max=8, min=3), IntSli…

In [8]:
POLIGONOS_REGULARES = {3: "triângulo", 4: "quadrado", 5: "pentágono", 6: "hexágono"}

total_de_possibilidades = 0
print(f"{'polígono':<12}{'ângulo interno':^18}{'faces possíveis num vértice':^30}")
for lados, nome in POLIGONOS_REGULARES.items():
    # Ângulo interno do polígono regular de n lados (módulo 04): 180° · (n − 2) / n.
    angulo_interno = 180 * (lados - 2) / lados
    # Testamos 3, 4, 5, ... faces enquanto a soma em volta do vértice for menor
    # que 360° (com menos de 3 faces não há ângulo poliédrico).
    possiveis = []
    faces = 3
    while faces * angulo_interno < 360:
        possiveis.append(faces)
        faces += 1
    total_de_possibilidades += len(possiveis)
    texto = ", ".join(str(k) for k in possiveis) or "nenhuma"
    print(f"{nome:<12}{f'{angulo_interno:.0f}°':^18}{texto:^30}")

print()
print(
    f"São só {total_de_possibilidades} possibilidades! Guarde esse número para o "
    "próximo notebook."
)

polígono      ângulo interno   faces possíveis num vértice  
triângulo          60°                   3, 4, 5            
quadrado           90°                      3               
pentágono          108°                     3               
hexágono           120°                  nenhuma            

São só 5 possibilidades! Guarde esse número para o próximo notebook.


**Pare e pense**

1. Por que seis triângulos equiláteros não formam a ponta de um poliedro?
2. Qual é o defeito no vértice de um cubo? E no vértice de um tetraedro regular?

<details>
<summary>Ver respostas</summary>

1. Porque $6 \times 60° = 360°$: as seis faces ficam planas, formando um hexágono (é o padrão de um piso de triângulos). Não sobra "folga" para dobrar.
2. Cubo: $360° - 3 \times 90° = 90°$. Tetraedro: $360° - 3 \times 60° = 180°$.
</details>

> 📎 **Conexão.** A tabela acima é a chave do notebook 07: cada uma das cinco possibilidades dá origem a exatamente um **poliedro de Platão**.

## Resumo

**Triedro** $V(a, b, c)$: três semirretas não coplanares de mesma origem. Seis elementos: faces $f_a = b\hat{V}c$, $f_b = a\hat{V}c$, $f_c = a\hat{V}b$ e diedros $d_a$, $d_b$, $d_c$.

**Faces**
- $f_a < f_b + f_c$ (e as outras duas)
- $f_a + f_b + f_c < 360°$
- Três ângulos que satisfazem essas condições sempre formam um triedro.

**Diedros**
- $180° < d_a + d_b + d_c < 540°$
- $d_b + d_c < 180° + d_a$ (e as outras duas)

**Congruência:** FDF, DFD, FFF e **DDD** (sem análogo nos triângulos). Triedros espelhados também são congruentes.

**Triedro polar:** $f'_a = 180° - d_a$ e $d'_a = 180° - f_a$. O polar do polar é o original. O trirretângulo é polar de si mesmo.

**Ângulo poliédrico convexo** com $n$ arestas: soma das faces $< 360°$; soma dos diedros entre $(n-2) \cdot 180°$ e $n \cdot 180°$. Defeito $= 360° - $ soma das faces.

**Polígonos regulares num vértice:** triângulos (3, 4 ou 5), quadrados (3), pentágonos (3). Hexágonos, nunca.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**.

O gabarito comentado está em [`0606b_triedros_e_angulos_poliedricos_gabarito.ipynb`](0606b_triedros_e_angulos_poliedricos_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Um triedro tem três faces e três diedros.
# (b) Existe triedro com faces 100°, 120° e 150°.
# (c) Existe triedro com faces 30°, 40° e 80°.
# (d) Em todo triedro, a soma dos diedros é maior que 180°.
# (e) Dois triedros com os três diedros respectivamente congruentes são
#     congruentes.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": True, "b": False, "c": False, "d": True, "e": True}
secoes_ex1 = {
    "a": "seção 1 - Triedro: conceitos e elementos",
    "b": "seção 2 - Relações entre as faces",
    "c": "seção 2 - Relações entre as faces",
    "d": "seção 4 - Triedros polares (ou suplementares)",
    "e": "seção 3 - Congruência de triedros",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; releia as definições "
        f"e os teoremas ({secoes_ex1[item]})."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: existe um triedro com estas faces? Troque cada ... por True
# ou False.
# (a) 90°, 90° e 90°
# (b) 60°, 60° e 60°
# (c) 120°, 120° e 120°
# (d) 50°, 60° e 100°
# (e) 50°, 60° e 120°
existe_ex2 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(existe_ex2)

In [ ]:
# Verificação
gabarito_ex2 = {"a": True, "b": True, "c": False, "d": True, "e": False}
for item, certa in gabarito_ex2.items():
    assert existe_ex2[item] == certa, (
        f"Tente de novo, revise: no item ({item}), confira as quatro condições: cada "
        "face menor que a soma das outras duas e a soma menor que 360° (seção 2 - "
        "Relações entre as faces)."
    )
print("Certo!")

# Conferência com o utils_3d: vertices_do_triedro só monta o triedro se ele existe.
casos_ex2 = {
    "a": (90, 90, 90),
    "b": (60, 60, 60),
    "c": (120, 120, 120),
    "d": (50, 60, 100),
    "e": (50, 60, 120),
}
for item, faces in casos_ex2.items():
    try:
        u3d.vertices_do_triedro(*faces)
        print(f"({item}) {faces}: o triedro existe.")
    except ValueError as erro:
        print(f"({item}) {faces}: {erro}")

In [ ]:
# Exercício Básico 3: triedro polar. Responda com listas de três números, em graus.
# (a) Um triedro tem diedros de 70°, 80° e 100°. Quais são as faces do seu polar,
#     na mesma ordem?
# (b) Um triedro tem faces de 60°, 60° e 60°. Quais são os diedros do seu polar?
faces_do_polar_ex3 = ...  # (a), por exemplo [10, 20, 30]
diedros_do_polar_ex3 = ...  # (b)

print(faces_do_polar_ex3, diedros_do_polar_ex3)

In [ ]:
# Verificação
assert (
    faces_do_polar_ex3 is not ...
    and len(faces_do_polar_ex3) == 3
    and all(
        abs(x - y) <= 0.1 for x, y in zip(faces_do_polar_ex3, [110, 100, 80])
    )
), (
    "Tente de novo, revise: face do polar = 180° − diedro do original, na mesma "
    "ordem (seção 4 - Triedros polares (ou suplementares))."
)
assert (
    diedros_do_polar_ex3 is not ...
    and len(diedros_do_polar_ex3) == 3
    and all(abs(x - 120) <= 0.1 for x in diedros_do_polar_ex3)
), (
    "Tente de novo, revise: diedro do polar = 180° − face do original (seção 4 - "
    "Triedros polares (ou suplementares))."
)
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: a barra de chocolate. Um triedro tem faces f_a = 90°,
# f_b = 90° e f_c = 60° (é o canto de uma barra de chocolate triangular). Quais são
# os diedros d_a, d_b e d_c, em graus?
diedros_ex4 = {
    "a": ...,
    "b": ...,
    "c": ...,
}

print(diedros_ex4)

In [ ]:
# Verificação
gabarito_ex4 = {"a": 90, "b": 90, "c": 60}
for item, certa in gabarito_ex4.items():
    resposta = diedros_ex4[item]
    assert resposta is not ... and abs(resposta - certa) <= 0.1, (
        f"Tente de novo, revise: no item ({item}), a aresta c forma 90° com a e com "
        "b, então ela é perpendicular ao plano da face f_c; use isso para achar a "
        "seção reta de d_c (seção 1 - Triedro: conceitos e elementos)."
    )
print("Certo!")

# Conferência com o utils_3d (que usa vetores, assunto do módulo 07).
t = u3d.vertices_do_triedro(90, 90, 60)
diedros = u3d.medir_elementos_do_triedro(t["V"], t["A"], t["B"], t["C"])["diedros"]
print({x: round(medida, 1) for x, medida in diedros.items()})
fig = u3d.criar_figura(titulo="O canto da barra de chocolate")
fig = u3d.desenhar_triedro(fig, t["V"], t["A"], t["B"], t["C"])
fig = u3d.desenhar_angulo(fig, t["V"], t["A"], t["B"], rotulo="60°", raio=1)
u3d.mostrar(fig)

In [ ]:
# Exercício Intermediário 5: num triedro, duas faces medem 70° e 50°. A terceira
# face só pode estar num intervalo aberto entre dois valores. Quais são esses dois
# valores? Responda com uma tupla (minimo, maximo).
intervalo_ex5 = ...

print(intervalo_ex5)

In [ ]:
# Verificação
assert (
    intervalo_ex5 is not ...
    and len(intervalo_ex5) == 2
    and abs(intervalo_ex5[0] - 20) <= 0.1
    and abs(intervalo_ex5[1] - 120) <= 0.1
), (
    "Tente de novo, revise: escreva as três desigualdades com a face desconhecida x "
    "e a condição da soma (seção 2 - Relações entre as faces)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 6: responda com números inteiros. No máximo, quantos
# (a) triângulos equiláteros, (b) quadrados, (c) pentágonos regulares e
# (d) hexágonos regulares podem formar, sozinhos, um ângulo poliédrico convexo?
# (Responda 0 se nenhuma quantidade for possível.)
maximo_ex6 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
}

print(maximo_ex6)

In [ ]:
# Verificação
gabarito_ex6 = {"a": 5, "b": 3, "c": 3, "d": 0}
for item, certa in gabarito_ex6.items():
    assert maximo_ex6[item] == certa, (
        f"Tente de novo, revise: no item ({item}), multiplique o ângulo interno pelo "
        "número de faces e compare com 360°; lembre que são necessárias pelo menos 3 "
        "faces (seção 5 - Ângulos poliédricos convexos)."
    )
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: triângulos sobre a Terra. Considere a Terra como uma esfera,
# com o vértice de um triedro no centro.
# (a) Um triângulo tem vértices no Polo Norte e em dois pontos do Equador separados
#     por 90° de longitude. Qual é a soma, em graus, dos seus três ângulos?
# (b) E se os dois pontos do Equador estiverem separados por 60° de longitude?
# Dica: os ângulos do triângulo esférico são os diedros do triedro. Quais são as
# faces desse triedro? (O ângulo entre o eixo da Terra e o plano do Equador é 90°.)
soma_ex7a = ...
soma_ex7b = ...

print(soma_ex7a, soma_ex7b)

In [ ]:
# Verificação
for resposta, certa, item in ((soma_ex7a, 270, "a"), (soma_ex7b, 240, "b")):
    assert resposta is not ... and abs(resposta - certa) <= 0.1, (
        f"Tente de novo, revise: no item ({item}), as faces são 90°, 90° e a "
        "diferença de longitude; os diedros nas arestas que vão ao Equador são retos, "
        "e o diedro na aresta do polo é igual à diferença de longitude (seção 4 - "
        "Triedros polares (ou suplementares))."
    )
print("Certo!")

# Conferência com o utils_3d: as arestas a e b vão ao Equador, e a aresta c, ao polo.
for item, longitude in (("a", 90), ("b", 60)):
    t = u3d.vertices_do_triedro(90, 90, longitude)
    diedros = u3d.medir_elementos_do_triedro(t["V"], t["A"], t["B"], t["C"])["diedros"]
    print(f"({item}) soma dos ângulos do triângulo: {sum(diedros.values()):.1f}°")
print(
    "A soma dos ângulos de um triângulo esférico passa de 180°; quanto maior o "
    "triângulo, maior o excesso."
)

In [ ]:
# Exercício Desafio 8: calcule, em graus, o diedro de um triedro com as três faces
# de 60° (o vértice de um tetraedro regular). Escreva o cálculo em Python.
# Dica: pense no tetraedro regular de aresta 1. No ponto médio M de uma aresta, os
# segmentos até os dois vértices opostos são alturas de triângulos equiláteros
# (h = √3/2) e formam a seção reta do diedro. O terceiro lado desse triângulo é
# uma aresta (1). Use a lei dos cossenos (e math.acos, math.degrees).
aresta = 1

diedro_ex8 = ...

print(diedro_ex8)

In [ ]:
# Verificação
assert diedro_ex8 is not ... and abs(diedro_ex8 - 70.5288) <= 0.1, (
    "Tente de novo, revise: cos θ = (h² + h² − 1²) / (2 · h · h), com h = √3/2 "
    "(seção 3 - Congruência de triedros)."
)
print("Certo!")

# Conferência com o utils_3d (que usa vetores, assunto do módulo 07).
t = u3d.vertices_do_triedro(60, 60, 60)
diedros = u3d.medir_elementos_do_triedro(t["V"], t["A"], t["B"], t["C"])["diedros"]
print(f"Diedro medido: {diedros['a']:.4f}°")

## Conexões

**Isso depende de:**
- [`0603a_perpendicularidade.ipynb`](0603a_perpendicularidade.ipynb) (planos perpendiculares, retas perpendiculares a planos)
- [`0605a_diedros.ipynb`](0605a_diedros.ipynb) (medida de diedros, seção reta, seções igualmente inclinadas)
- [`0404a_triangulos_conceito_elementos_e_classificacao.ipynb`](../04_geometria_plana/0404a_triangulos_conceito_elementos_e_classificacao.ipynb) (desigualdade triangular)
- [`0405a_congruencia_de_triangulos.ipynb`](../04_geometria_plana/0405a_congruencia_de_triangulos.ipynb) (casos de congruência de triângulos)
- [`0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb`](../04_geometria_plana/0418a_poligonos_regulares_e_comprimento_da_circunferencia.ipynb) (polígonos regulares e seus ângulos internos)
- [`0510a_triangulos_quaisquer.ipynb`](../05_trigonometria/0510a_triangulos_quaisquer.ipynb) (lei dos cossenos)

**Isso será usado em:**
- 📌 Poliedros convexos (vértices dos poliedros, os cinco poliedros de Platão e a soma dos ângulos das faces)
- 📌 Pirâmide (o vértice da pirâmide é um ângulo poliédrico)
- 📌 Esfera (triângulos esféricos)
- 📌 Superfícies e sólidos esféricos (triângulos esféricos, fuso e cunha)